# EXP-QPU-CQRF-CBR-002-QPU150 — IBM QPU inference spot-check

Inference-only hardware feasibility notebook for **CQRF + CBR**, using **150 held-out samples per dataset**.

- CQRF: 10-qubit Coupled Quantum Reservoir Features, 57 observables.
- CBR: Compact Bottleneck Readout, code name TinyMLP2, architecture `57 -> 2 -> 2 -> 1`, 125 trainable parameters.
- QPU is used only for CQRF feature extraction on unseen test samples.
- The readout is trained on simulator CQRF features and frozen before QPU inference.
- Ridge-control is optional and uses the same QPU features, so it adds no extra QPU circuits.

QPU scope:
- TJU: 150 samples = 450 circuits
- MIT: 150 samples = 450 circuits
- HUST: 150 samples = 450 circuits
- Total: 450 samples = 1350 circuits

**Default is safe:** `RUN_QPU = False`. First run local dry-run, then set `RUN_QPU = True`.


In [ ]:
# Install dependencies
!pip -q install qiskit qiskit-ibm-runtime qiskit-aer


In [ ]:
# Imports, Drive mount, and configuration
import os, json, time, datetime, warnings, getpass
from pathlib import Path
import numpy as np
import pandas as pd

# Mount Google Drive in Colab. This notebook needs files from MyDrive.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print('Drive mount skipped or unavailable:', e)

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

from qiskit import QuantumCircuit, ClassicalRegister
from qiskit.quantum_info import Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/content/drive/MyDrive/HB/QRC_Battery_SOH')
EXP_ID = 'EXP-QPU-CQRF-CBR-002-QPU150'

# Main selected feature folder prepared before QPU.
FEATURE_DIR = PROJECT_ROOT / 'features' / 'CQRF-CBR'

# If the exact folder is not visible, search for it under MyDrive.
# This handles stale Drive mounts, accidental spaces, or folder placement issues.
required_names = [
    # Base simulator feature bundles
    'CQRF-c0_3_Dataset_1_NCA_battery_seed0.npz',
    'CQRF-c0_3_Dataset_2_NCM_battery_seed0.npz',
    'CQRF-c0_3_Dataset_3_NCM_NCA_battery_seed0.npz',
    'CQRF-c0_3_MIT_seed0.npz',
    'CQRF-c0_3_HUST_seed0_reconstructed_full.npz',

    # Reconstructed full packages used to build QPU-150 selections
    'CQRF-c0_3_Dataset_1_NCA_battery_seed0_reconstructed_full.npz',
    'CQRF-c0_3_Dataset_2_NCM_battery_seed0_reconstructed_full.npz',
    'CQRF-c0_3_Dataset_3_NCM_NCA_battery_seed0_reconstructed_full.npz',
    'CQRF-c0_3_MIT_seed0_reconstructed_full.npz',
]

def folder_has_required_files(folder: Path) -> bool:
    return folder.exists() and all((folder / name).exists() for name in required_names)

if not folder_has_required_files(FEATURE_DIR):
    print('Exact FEATURE_DIR not ready:', FEATURE_DIR)
    print('Searching for a folder containing the required CQRF-CBR files...')
    candidates = []
    search_roots = [PROJECT_ROOT / 'features', Path('/content/drive/MyDrive')]
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        for f in root.rglob('CQRF-c0_3_MIT_seed0_reconstructed_full.npz'):
            cand = f.parent
            if cand in seen:
                continue
            seen.add(cand)
            ok_count = sum((cand / name).exists() for name in required_names)
            candidates.append((ok_count, cand))
    candidates = sorted(candidates, key=lambda x: (-x[0], str(x[1])))
    print('Candidates found:')
    for ok_count, cand in candidates[:10]:
        print(f'  {ok_count}/{len(required_names)} -> {cand}')
    if candidates and candidates[0][0] == len(required_names):
        FEATURE_DIR = candidates[0][1]
        print('Using discovered FEATURE_DIR:', FEATURE_DIR)
    else:
        print('\nFolders currently visible under PROJECT_ROOT/features:')
        feat_root = PROJECT_ROOT / 'features'
        if feat_root.exists():
            for p in sorted(feat_root.iterdir()):
                print(' ', repr(p.name))
        raise FileNotFoundError(
            'Could not locate a CQRF-CBR feature folder containing all required files. '
            'Mount Drive, then ensure features/CQRF-CBR contains the 8 required .npz files.'
        )

RESULT_DIR = PROJECT_ROOT / 'results' / EXP_ID
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / EXP_ID
QPU_FEATURE_DIR = PROJECT_ROOT / 'features' / EXP_ID
LOG_DIR = PROJECT_ROOT / 'logs' / EXP_ID
FIG_DIR = PROJECT_ROOT / 'figures' / EXP_ID
for d in [RESULT_DIR, CHECKPOINT_DIR, QPU_FEATURE_DIR, LOG_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Run controls
SHOTS = 512
RUN_LOCAL_DRY_RUN = False
RUN_QPU = False          # Keep False until dry run passes. Then set True.
DRY_RUN_SAMPLES_PER_DATASET = 2
QPU_BATCH_CIRCUITS = 60
RUN_RIDGE_CONTROL = True # No extra QPU circuits; same QPU-derived features.

# CQRF constants
CQRF_SEED = 42
N_AUX = 4
N_PROC = 6
N_QUBITS = N_AUX + N_PROC
N_LAYERS = 2
LAMBDA_DIRECT = 0.30
LAMBDA_REVERSE = 0.15
LAMBDA_SKIP = 0.09
AUX_WIRES = list(range(N_AUX))
PROC_WIRES = list(range(N_AUX, N_QUBITS))
PROC_PAIRS = [(i, j) for i in PROC_WIRES for j in PROC_WIRES if i < j]
CROSS_PAIRS = [(i, j) for i in AUX_WIRES for j in PROC_WIRES]
N_OUTPUTS = 57

# CBR controls
CBR_EPOCHS = 300
CBR_LR = 1e-3
CBR_WEIGHT_DECAY = 1e-5
CBR_BATCH_SIZE = 64
CBR_PATIENCE = 40
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FEATURE_DIR:', FEATURE_DIR)
print('FEATURE_DIR exists:', FEATURE_DIR.exists())
print('Required file check:')
for name in required_names:
    print(' ', (FEATURE_DIR / name).exists(), name)
print('DEVICE:', DEVICE)
print('RUN_QPU:', RUN_QPU, '| SHOTS:', SHOTS)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/HB/QRC_Battery_SOH
FEATURE_DIR: /content/drive/MyDrive/HB/QRC_Battery_SOH/features/CQRF-CBR
FEATURE_DIR exists: True
Required file check:
  True CQRF-c0_3_Dataset_1_NCA_battery_seed0.npz
  True CQRF-c0_3_Dataset_2_NCM_battery_seed0.npz
  True CQRF-c0_3_Dataset_3_NCM_NCA_battery_seed0.npz
  True CQRF-c0_3_MIT_seed0.npz
  True CQRF-c0_3_HUST_seed0_reconstructed_full.npz
  True CQRF-c0_3_Dataset_1_NCA_battery_seed0_reconstructed_full.npz
  True CQRF-c0_3_Dataset_2_NCM_battery_seed0_reconstructed_full.npz
  True CQRF-c0_3_Dataset_3_NCM_NCA_battery_seed0_reconstructed_full.npz
  True CQRF-c0_3_MIT_seed0_reconstructed_full.npz
DEVICE: cpu
RUN_QPU: True | SHOTS: 512


In [ ]:
# Utility functions
def log(msg):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(LOG_DIR / f'{EXP_ID}_log.txt', 'a') as f:
        f.write(line + '\n')

def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))))

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mm = mape(y_true, y_pred)
    return {
        'MAPE': mm,
        'RMSE': rmse(y_true, y_pred),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        'DerivedScore': float(100 * (1 - mm)),
    }

def seed_everything(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def fit_feature_scaler(R):
    mean = R.mean(axis=0).astype(np.float32)
    std = R.std(axis=0).astype(np.float32)
    std = np.where(std < 1e-8, 1.0, std).astype(np.float32)
    return {'mean': mean, 'std': std}

def transform_features(R, st):
    return ((R - st['mean']) / st['std']).astype(np.float32)


In [ ]:
# CBR readout and optional Ridge control
class CBR(nn.Module):
    # Compact Bottleneck Readout: 57 -> 2 -> 2 -> 1, 125 params.
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(57, 2), nn.Tanh(),
            nn.Linear(2, 2), nn.Tanh(),
            nn.Linear(2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

assert sum(p.numel() for p in CBR().parameters() if p.requires_grad) == 125

def train_cbr(Rtr, ytr, Rv, yv, seed=0, name='model'):
    seed_everything(seed)
    st = fit_feature_scaler(Rtr)
    Xtr = torch.tensor(transform_features(Rtr, st), dtype=torch.float32, device=DEVICE)
    ytr_t = torch.tensor(ytr.astype(np.float32), dtype=torch.float32, device=DEVICE)
    Xv = torch.tensor(transform_features(Rv, st), dtype=torch.float32, device=DEVICE)
    yv_t = torch.tensor(yv.astype(np.float32), dtype=torch.float32, device=DEVICE)
    model = CBR().to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=CBR_LR, weight_decay=CBR_WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    best_val, best_state, best_epoch = float('inf'), None, 0
    patience_left = CBR_PATIENCE
    n = len(Xtr)
    for ep in range(1, CBR_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for s in range(0, n, CBR_BATCH_SIZE):
            idx = perm[s:s+CBR_BATCH_SIZE]
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(Xtr[idx]), ytr_t[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(model(Xv), yv_t).detach().cpu())
        if val_loss < best_val - 1e-10:
            best_val, best_epoch = val_loss, ep
            patience_left = CBR_PATIENCE
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_left -= 1
            if patience_left <= 0:
                break
    model.load_state_dict(best_state)
    model.eval()
    ckpt_path = CHECKPOINT_DIR / f'CQRF_CBR_{name}.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'feature_scaler_mean': st['mean'],
        'feature_scaler_std': st['std'],
        'architecture': [57, 2, 2, 1],
        'readout_name': 'CBR',
        'name': name,
        'seed': seed,
        'best_val_loss': best_val,
        'best_epoch': best_epoch,
    }, ckpt_path)
    return model, st, {'best_val_loss': best_val, 'best_epoch': best_epoch, 'ckpt': str(ckpt_path)}

def predict_cbr(model, st, R):
    X = torch.tensor(transform_features(R, st), dtype=torch.float32, device=DEVICE)
    outs = []
    model.eval()
    with torch.no_grad():
        for s in range(0, len(X), 4096):
            outs.append(model(X[s:s+4096]).detach().cpu().numpy())
    return np.concatenate(outs).astype(np.float32)

class SimpleRidgeControl:
    def __init__(self, alpha=1.0):
        self.scaler = None
        self.model = Ridge(alpha=alpha)
    def fit(self, R, y):
        self.scaler = fit_feature_scaler(R)
        self.model.fit(transform_features(R, self.scaler), y)
    def predict(self, R):
        return self.model.predict(transform_features(R, self.scaler)).astype(np.float32)


In [ ]:

# Create QPU-150 selection files from reconstructed full packages
# This cell does not overwrite QPU-150 files. It creates separate qpu150 files.

def linspace_idx(n, k):
    return np.linspace(0, n - 1, k, dtype=np.int64)

def load_npz_checked(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return np.load(path, allow_pickle=True)

print("Creating/refreshing QPU-150 selection files in:", FEATURE_DIR)

# MIT qpu150
mit = load_npz_checked(FEATURE_DIR / "CQRF-c0_3_MIT_seed0_reconstructed_full.npz")
idx = linspace_idx(len(mit["yte"]), 150)

np.savez_compressed(
    FEATURE_DIR / "CQRF-c0_3_MIT_seed0_qpu150_selection.npz",
    idx=idx,
    R_sim=mit["Rte"][idx].astype(np.float32),
    y=mit["yte"][idx].astype(np.float32),
    X_norm=mit["Xte_norm"][idx].astype(np.float32),
    cell_ids=mit["gte"][idx].astype(object),
    split_seed=np.array(0),
    warmup=mit["warmup"],
    model_name=mit["model_name"],
    readout_name=np.array("CBR"),
)
print("Saved MIT qpu150")

# HUST qpu150
hust = load_npz_checked(FEATURE_DIR / "CQRF-c0_3_HUST_seed0_reconstructed_full.npz")
idx = linspace_idx(len(hust["yte"]), 150)

np.savez_compressed(
    FEATURE_DIR / "CQRF-c0_3_HUST_seed0_qpu150_selection.npz",
    idx=idx,
    R_sim=hust["Rte"][idx].astype(np.float32),
    y=hust["yte"][idx].astype(np.float32),
    X_norm=hust["Xte_norm"][idx].astype(np.float32),
    cell_ids=hust["gte"][idx].astype(object),
    split_seed=np.array(0),
    warmup=hust["warmup"],
    model_name=hust["model_name"],
    readout_name=np.array("CBR"),
)
print("Saved HUST qpu150")

# TJU qpu150: 50 samples from each of the 3 TJU subsets
tju_files = [
    ("Dataset_1_NCA_battery", FEATURE_DIR / "CQRF-c0_3_Dataset_1_NCA_battery_seed0_reconstructed_full.npz"),
    ("Dataset_2_NCM_battery", FEATURE_DIR / "CQRF-c0_3_Dataset_2_NCM_battery_seed0_reconstructed_full.npz"),
    ("Dataset_3_NCM_NCA_battery", FEATURE_DIR / "CQRF-c0_3_Dataset_3_NCM_NCA_battery_seed0_reconstructed_full.npz"),
]

all_R, all_y, all_X, all_cells, all_subset, all_idx = [], [], [], [], [], []

for subset_name, path in tju_files:
    z = load_npz_checked(path)
    idx = linspace_idx(len(z["yte"]), 50)

    all_R.append(z["Rte"][idx].astype(np.float32))
    all_y.append(z["yte"][idx].astype(np.float32))
    all_X.append(z["Xte_norm"][idx].astype(np.float32))
    all_cells.append(z["gte"][idx].astype(object))
    all_subset.append(np.array([subset_name] * len(idx), dtype=object))
    all_idx.append(idx.astype(np.int64))

R_sim = np.vstack(all_R).astype(np.float32)
y = np.concatenate(all_y).astype(np.float32)
X_norm = np.vstack(all_X).astype(np.float32)
cell_ids = np.concatenate(all_cells).astype(object)
subset_labels = np.concatenate(all_subset).astype(object)
local_idx = np.concatenate(all_idx).astype(np.int64)

assert R_sim.shape == (150, 57), R_sim.shape
assert X_norm.shape == (150, 6), X_norm.shape
assert y.shape == (150,), y.shape

np.savez_compressed(
    FEATURE_DIR / "CQRF-c0_3_TJU_seed0_qpu150_selection.npz",
    local_idx=local_idx,
    R_sim=R_sim,
    y=y,
    X_norm=X_norm,
    cell_ids=cell_ids,
    subset_labels=subset_labels,
    split_seed=np.array(0),
    warmup=np.array(3),
    model_name=np.array("CQRF-c0.3"),
    readout_name=np.array("CBR"),
)
print("Saved TJU qpu150")

# Final selection check
for name in [
    "CQRF-c0_3_TJU_seed0_qpu150_selection.npz",
    "CQRF-c0_3_MIT_seed0_qpu150_selection.npz",
    "CQRF-c0_3_HUST_seed0_qpu150_selection.npz",
]:
    z = np.load(FEATURE_DIR / name, allow_pickle=True)
    print("\n", name)
    print("  R_sim :", z["R_sim"].shape)
    print("  X_norm:", z["X_norm"].shape)
    print("  y     :", z["y"].shape)


Creating/refreshing QPU-150 selection files in: /content/drive/MyDrive/HB/QRC_Battery_SOH/features/CQRF-CBR
Saved MIT qpu150
Saved HUST qpu150
Saved TJU qpu150

 CQRF-c0_3_TJU_seed0_qpu150_selection.npz
  R_sim : (150, 57)
  X_norm: (150, 6)
  y     : (150,)

 CQRF-c0_3_MIT_seed0_qpu150_selection.npz
  R_sim : (150, 57)
  X_norm: (150, 8)
  y     : (150,)

 CQRF-c0_3_HUST_seed0_qpu150_selection.npz
  R_sim : (150, 57)
  X_norm: (150, 6)
  y     : (150,)


In [ ]:
# Load simulator feature bundles and QPU-150 selections
FILES = {
    'TJU_NCA': FEATURE_DIR / 'CQRF-c0_3_Dataset_1_NCA_battery_seed0.npz',
    'TJU_NCM': FEATURE_DIR / 'CQRF-c0_3_Dataset_2_NCM_battery_seed0.npz',
    'TJU_BLEND': FEATURE_DIR / 'CQRF-c0_3_Dataset_3_NCM_NCA_battery_seed0.npz',
    'MIT': FEATURE_DIR / 'CQRF-c0_3_MIT_seed0.npz',
    'HUST': FEATURE_DIR / 'CQRF-c0_3_HUST_seed0_reconstructed_full.npz',
}
SEL_FILES = {
    'TJU': FEATURE_DIR / 'CQRF-c0_3_TJU_seed0_qpu150_selection.npz',
    'MIT': FEATURE_DIR / 'CQRF-c0_3_MIT_seed0_qpu150_selection.npz',
    'HUST': FEATURE_DIR / 'CQRF-c0_3_HUST_seed0_qpu150_selection.npz',
}
SUBSET_TO_MODEL = {
    'Dataset_1_NCA_battery': 'TJU_NCA',
    'Dataset_2_NCM_battery': 'TJU_NCM',
    'Dataset_3_NCM_NCA_battery': 'TJU_BLEND',
}
for k, p in FILES.items(): print(k, p.exists(), p.name)
for k, p in SEL_FILES.items(): print('SEL', k, p.exists(), p.name)

def load_bundle(key):
    z = np.load(FILES[key], allow_pickle=True)
    return {
        'Rtr': z['Rtr'].astype(np.float32), 'ytr': z['ytr'].astype(np.float32),
        'Rv': z['Rv'].astype(np.float32), 'yv': z['yv'].astype(np.float32),
        'Rte': z['Rte'].astype(np.float32), 'yte': z['yte'].astype(np.float32),
    }

bundles = {k: load_bundle(k) for k in FILES}
selections = {k: np.load(p, allow_pickle=True) for k, p in SEL_FILES.items()}
for k, b in bundles.items():
    print('\nBundle', k, 'train', b['Rtr'].shape, b['ytr'].shape, 'valid', b['Rv'].shape, 'test', b['Rte'].shape)
for k, z in selections.items():
    print('\nSelection', k, 'R_sim', z['R_sim'].shape, 'X_norm', z['X_norm'].shape, 'y', z['y'].shape)


TJU_NCA True CQRF-c0_3_Dataset_1_NCA_battery_seed0.npz
TJU_NCM True CQRF-c0_3_Dataset_2_NCM_battery_seed0.npz
TJU_BLEND True CQRF-c0_3_Dataset_3_NCM_NCA_battery_seed0.npz
MIT True CQRF-c0_3_MIT_seed0.npz
HUST True CQRF-c0_3_HUST_seed0_reconstructed_full.npz
SEL TJU True CQRF-c0_3_TJU_seed0_qpu150_selection.npz
SEL MIT True CQRF-c0_3_MIT_seed0_qpu150_selection.npz
SEL HUST True CQRF-c0_3_HUST_seed0_qpu150_selection.npz

Bundle TJU_NCA train (12689, 57) (12689,) valid (4766, 57) test (5449, 57)

Bundle TJU_NCM train (15665, 57) (15665,) valid (4458, 57) test (5925, 57)

Bundle TJU_BLEND train (4813, 57) (4813,) valid (989, 57) test (2953, 57)

Bundle MIT train (69892, 57) (69892,) valid (23336, 57) test (21449, 57)

Bundle HUST train (88271, 57) (88271,) valid (26931, 57) test (30905, 57)

Selection TJU R_sim (150, 57) X_norm (150, 6) y (150,)

Selection MIT R_sim (150, 57) X_norm (150, 8) y (150,)

Selection HUST R_sim (150, 57) X_norm (150, 6) y (150,)


In [ ]:
# Train CBR models from simulator CQRF features and sanity-check full test metrics
models, ridge_models, full_test_rows = {}, {}, []
for key, b in bundles.items():
    log(f'Training CBR for {key}...')
    model, st, info = train_cbr(b['Rtr'], b['ytr'], b['Rv'], b['yv'], seed=0, name=key)
    yhat = predict_cbr(model, st, b['Rte'])
    met = compute_metrics(b['yte'], yhat)
    models[key] = {'model': model, 'scaler': st, 'info': info}
    full_test_rows.append({'model_key': key, **met, 'best_epoch': info['best_epoch'], 'val_loss': info['best_val_loss']})
    print(f"{key}: MAPE={met['MAPE']:.6f} RMSE={met['RMSE']:.6f} Score={met['DerivedScore']:.3f} epoch={info['best_epoch']}")
    if RUN_RIDGE_CONTROL:
        rr = SimpleRidgeControl(alpha=1.0)
        rr.fit(b['Rtr'], b['ytr'])
        ridge_models[key] = rr
full_test_df = pd.DataFrame(full_test_rows)
full_test_df.to_csv(RESULT_DIR / 'cbr_full_test_sanity_metrics.csv', index=False)
full_test_df


[18:26:51] Training CBR for TJU_NCA...
TJU_NCA: MAPE=0.049898 RMSE=0.050559 Score=95.010 epoch=62
[18:27:17] Training CBR for TJU_NCM...
TJU_NCM: MAPE=0.025048 RMSE=0.016435 Score=97.495 epoch=76
[18:27:38] Training CBR for TJU_BLEND...
TJU_BLEND: MAPE=0.003262 RMSE=0.004575 Score=99.674 epoch=236
[18:27:54] Training CBR for MIT...
MIT: MAPE=0.015393 RMSE=0.021531 Score=98.461 epoch=85
[18:29:37] Training CBR for HUST...
HUST: MAPE=0.022154 RMSE=0.027429 Score=97.785 epoch=56


,model_key,MAPE,RMSE,MAE,R2,DerivedScore,best_epoch,val_loss
0,TJU_NCA,0.049898,0.050559,0.018162,0.806864,95.010199,62,0.000364
1,TJU_NCM,0.025048,0.016435,0.010164,0.978807,97.495156,76,0.000068
2,TJU_BLEND,0.003262,0.004575,0.002641,0.997843,99.673781,236,0.000012
3,MIT,0.015393,0.021531,0.014419,0.800312,98.460710,85,0.000315
4,HUST,0.022154,0.027429,0.019351,0.863330,97.784637,56,0.000926


In [ ]:
# Predict selected simulator features using frozen CBR models
def predict_selection_with_cbr(dataset_name, R):
    if dataset_name == 'TJU':
        labels = selections['TJU']['subset_labels'].astype(object)
        out = np.zeros(len(R), dtype=np.float32)
        for subset_label, model_key in SUBSET_TO_MODEL.items():
            mask = np.array([str(x) == subset_label for x in labels])
            if mask.any():
                m = models[model_key]
                out[mask] = predict_cbr(m['model'], m['scaler'], R[mask])
        return out
    m = models[dataset_name]
    return predict_cbr(m['model'], m['scaler'], R)

def predict_selection_with_ridge(dataset_name, R):
    if not RUN_RIDGE_CONTROL: return None
    if dataset_name == 'TJU':
        labels = selections['TJU']['subset_labels'].astype(object)
        out = np.zeros(len(R), dtype=np.float32)
        for subset_label, model_key in SUBSET_TO_MODEL.items():
            mask = np.array([str(x) == subset_label for x in labels])
            if mask.any(): out[mask] = ridge_models[model_key].predict(R[mask])
        return out
    return ridge_models[dataset_name].predict(R)

selection_rows = []
for ds in ['TJU','MIT','HUST']:
    z = selections[ds]
    R, y = z['R_sim'].astype(np.float32), z['y'].astype(np.float32)
    pred = predict_selection_with_cbr(ds, R)
    met = compute_metrics(y, pred)
    selection_rows.append({'dataset': ds, 'source': 'precomputed_simulator_features', 'readout': 'CBR', **met})
    print(f"{ds} selected SIM-CBR: MAPE={met['MAPE']:.6f} RMSE={met['RMSE']:.6f} Score={met['DerivedScore']:.3f}")
    if RUN_RIDGE_CONTROL:
        pred_r = predict_selection_with_ridge(ds, R)
        met_r = compute_metrics(y, pred_r)
        selection_rows.append({'dataset': ds, 'source': 'precomputed_simulator_features', 'readout': 'Ridge-control', **met_r})
        print(f"{ds} selected SIM-Ridge: MAPE={met_r['MAPE']:.6f} RMSE={met_r['RMSE']:.6f} Score={met_r['DerivedScore']:.3f}")
sel_df = pd.DataFrame(selection_rows)
sel_df.to_csv(RESULT_DIR / 'selected_simulator_metrics.csv', index=False)
sel_df


TJU selected SIM-CBR: MAPE=0.127309 RMSE=0.028743 Score=87.269
TJU selected SIM-Ridge: MAPE=0.298036 RMSE=0.098916 Score=70.196
MIT selected SIM-CBR: MAPE=0.015101 RMSE=0.021208 Score=98.490
MIT selected SIM-Ridge: MAPE=0.016025 RMSE=0.021858 Score=98.397
HUST selected SIM-CBR: MAPE=0.021068 RMSE=0.025324 Score=97.893
HUST selected SIM-Ridge: MAPE=0.023838 RMSE=0.029259 Score=97.616


,dataset,source,readout,MAPE,RMSE,MAE,R2,DerivedScore
0,TJU,precomputed_simulator_features,CBR,0.127309,0.028743,0.010042,0.926613,87.269135
1,TJU,precomputed_simulator_features,Ridge-control,0.298036,0.098916,0.023685,0.130883,70.196381
2,MIT,precomputed_simulator_features,CBR,0.015101,0.021208,0.014081,0.801971,98.489877
3,MIT,precomputed_simulator_features,Ridge-control,0.016025,0.021858,0.014986,0.789647,98.397464
4,HUST,precomputed_simulator_features,CBR,0.021068,0.025324,0.018488,0.884020,97.893156
5,HUST,precomputed_simulator_features,Ridge-control,0.023838,0.029259,0.020815,0.845173,97.616229


In [ ]:

# CQRF Qiskit circuit and feature extraction
# Patched version:
#   - TJU/MIT use the generic CQRF builder from the existing QPU notebook.
#   - HUST uses a HUST-specific CQRF builder matching EXP_CQRF_TinyMLP2_HUST_005.
#
# Do not run IBM QPU until the local dry-run passes for TJU, MIT, and HUST.

# -----------------------------
# Generic builder: keep for TJU/MIT
# -----------------------------
def get_cqrf_params(n_inputs, seed=CQRF_SEED):
    rng = np.random.RandomState(seed)
    reservoir_params = rng.uniform(0, 2*np.pi, size=(N_LAYERS, N_QUBITS, 2))
    input_projection = rng.randn(n_inputs, N_PROC) * 0.5
    aux_rotation = rng.uniform(0, 2*np.pi, size=(N_AUX, 2))
    return (
        reservoir_params.astype(np.float64),
        input_projection.astype(np.float64),
        aux_rotation.astype(np.float64),
    )

def build_cqrf_unitary_circuit_generic(x_norm, n_inputs):
    reservoir_params, input_projection, aux_rotation = get_cqrf_params(n_inputs)
    projected = np.clip(np.asarray(x_norm, dtype=np.float64) @ input_projection, -np.pi, np.pi)

    qc = QuantumCircuit(N_QUBITS)

    # Static auxiliary register, generic notebook version
    for i, w in enumerate(AUX_WIRES):
        qc.ry(0.0, w)
        qc.rz(float(aux_rotation[i, 0]), w)

    # Processor input encoding
    for i, w in enumerate(PROC_WIRES):
        qc.ry(float(projected[i]), w)
        qc.rz(float(projected[i] * 0.5), w)

    # Intra-register entanglement
    for i in range(N_AUX - 1):
        qc.cx(AUX_WIRES[i], AUX_WIRES[i + 1])

    for i in range(N_PROC - 1):
        qc.cx(PROC_WIRES[i], PROC_WIRES[i + 1])
    qc.cx(PROC_WIRES[-1], PROC_WIRES[0])

    # Cross-register CQRF coupling
    for i in range(N_AUX):
        qc.cry(float(LAMBDA_DIRECT * np.pi), AUX_WIRES[i], PROC_WIRES[i])
        qc.cry(float(LAMBDA_REVERSE * np.pi), PROC_WIRES[i], AUX_WIRES[i])

    for i in range(N_AUX):
        qc.cry(float(LAMBDA_SKIP * np.pi), AUX_WIRES[i], PROC_WIRES[(i + 2) % N_PROC])

    # Fixed reservoir layers
    for layer in range(N_LAYERS):
        for i in range(N_QUBITS):
            qc.ry(float(reservoir_params[layer, i, 0]), i)
            qc.rz(float(reservoir_params[layer, i, 1]), i)

        for i in range(N_QUBITS):
            qc.cx(i, (i + 1) % N_QUBITS)

        for i in range(0, N_QUBITS - 2, 2):
            qc.cx(i, i + 2)

    return qc


# -----------------------------
# HUST-specific builder
# This matches the HUST CQRF/TinyMLP2 feature-generation notebook:
#   - numpy default_rng seeds
#   - fixed auxiliary RY/RZ
#   - projection seed = 1234 + input_dim
#   - all i -> i+2 skip CNOTs inside reservoir layer
# -----------------------------
HUST_CIRCUIT_SEED = 42
HUST_PROJECTION_SEED = 1234
HUST_COUPLING_STRENGTH = 0.3

_hust_rng = np.random.default_rng(HUST_CIRCUIT_SEED)
HUST_FIX_MEM_RY = _hust_rng.uniform(-np.pi, np.pi, 4)
HUST_FIX_MEM_RZ = _hust_rng.uniform(-np.pi, np.pi, 4)
HUST_RES_THETA = _hust_rng.uniform(0, 2*np.pi, (2, 10))
HUST_RES_PHI = _hust_rng.uniform(0, 2*np.pi, (2, 10))

def hust_make_projection(d):
    r = np.random.default_rng(HUST_PROJECTION_SEED + int(d))
    return r.normal(0, 1 / np.sqrt(d), (d, 6)).astype(np.float64)

def build_cqrf_unitary_circuit_hust(x_norm, n_inputs):
    W = hust_make_projection(n_inputs)
    projected = np.clip(np.asarray(x_norm, dtype=np.float64) @ W, -np.pi, np.pi)

    qc = QuantumCircuit(N_QUBITS)

    # Fixed auxiliary register. No recurrent feedback in CQRF.
    for i, w in enumerate(AUX_WIRES):
        qc.ry(float(HUST_FIX_MEM_RY[i]), w)
        qc.rz(float(HUST_FIX_MEM_RZ[i]), w)

    # Processor input encoding.
    for j, w in enumerate(PROC_WIRES):
        qc.ry(float(projected[j]), w)
        qc.rz(float(projected[j]) / 2.0, w)

    # Intra-register entanglement.
    for a, b in zip(AUX_WIRES[:-1], AUX_WIRES[1:]):
        qc.cx(a, b)

    for a, b in zip(PROC_WIRES, PROC_WIRES[1:] + [PROC_WIRES[0]]):
        qc.cx(a, b)

    # Coupled cross-register interactions.
    c = HUST_COUPLING_STRENGTH
    for i, mw in enumerate(AUX_WIRES):
        pw = PROC_WIRES[i]
        qc.cry(float(c * np.pi), mw, pw)
        qc.cry(float(c * np.pi / 2.0), pw, mw)
        qc.cry(float(0.3 * c * np.pi), mw, PROC_WIRES[(i + 2) % 6])

    # Fixed reservoir layers.
    for layer in range(2):
        for i, w in enumerate(range(N_QUBITS)):
            qc.ry(float(HUST_RES_THETA[layer, i]), w)
            qc.rz(float(HUST_RES_PHI[layer, i]), w)

        for i in range(N_QUBITS):
            qc.cx(i, (i + 1) % N_QUBITS)

        # Important HUST detail: all i -> i+2, not only even i.
        for i in range(N_QUBITS):
            qc.cx(i, (i + 2) % N_QUBITS)

    return qc


def build_cqrf_unitary_circuit_for_dataset(dataset_name, x_norm, n_inputs):
    if str(dataset_name).upper() == "HUST":
        return build_cqrf_unitary_circuit_hust(x_norm, n_inputs)
    return build_cqrf_unitary_circuit_generic(x_norm, n_inputs)


# -----------------------------
# Measurement helpers
# -----------------------------
def basis_rotated_unitary(qc_unitary, basis):
    qc = qc_unitary.copy()

    if basis == "X":
        for w in PROC_WIRES:
            qc.h(w)

    elif basis == "Y":
        for w in PROC_WIRES:
            qc.sdg(w)
            qc.h(w)

    elif basis != "Z":
        raise ValueError(basis)

    return qc

def add_basis_and_measure(qc_unitary, basis):
    qc = basis_rotated_unitary(qc_unitary, basis)
    meas = ClassicalRegister(N_QUBITS, "meas")
    qc.add_register(meas)
    qc.measure(list(range(N_QUBITS)), meas)
    return qc

def _bits_little_endian(bitstring):
    s = str(bitstring).replace(" ", "")
    bits = [int(b) for b in s[::-1]]
    if len(bits) < N_QUBITS:
        bits += [0] * (N_QUBITS - len(bits))
    return bits[:N_QUBITS]

def expectation_from_distribution(dist, wires):
    total = float(sum(dist.values()))
    acc = 0.0

    for bitstring, val in dist.items():
        bits = _bits_little_endian(bitstring)
        prod = 1.0
        for w in wires:
            prod *= (1.0 - 2.0 * bits[w])
        acc += prod * float(val)

    return acc / total

def features_from_basis_distributions(z_dist, x_dist, y_dist):
    feats = []

    for w in PROC_WIRES:
        feats.extend([
            expectation_from_distribution(x_dist, [w]),
            expectation_from_distribution(y_dist, [w]),
            expectation_from_distribution(z_dist, [w]),
        ])

    for (i, j) in PROC_PAIRS:
        feats.append(expectation_from_distribution(z_dist, [i, j]))

    for (i, j) in CROSS_PAIRS:
        feats.append(expectation_from_distribution(z_dist, [i, j]))

    return np.asarray(feats, dtype=np.float32)

def local_statevector_cqrf_features(dataset_name, X_norm):
    n_inputs = X_norm.shape[1]
    out = []

    for x in X_norm:
        qc0 = build_cqrf_unitary_circuit_for_dataset(dataset_name, x, n_inputs)
        dists = {}

        for basis in ["Z", "X", "Y"]:
            dists[basis] = Statevector.from_instruction(
                basis_rotated_unitary(qc0, basis)
            ).probabilities_dict()

        out.append(features_from_basis_distributions(dists["Z"], dists["X"], dists["Y"]))

    return np.vstack(out).astype(np.float32)

def make_qpu_circuits_for_dataset(dataset_name, X_norm):
    n_inputs = X_norm.shape[1]
    circuits, meta = [], []

    for i, x in enumerate(X_norm):
        qc0 = build_cqrf_unitary_circuit_for_dataset(dataset_name, x, n_inputs)

        for basis in ["Z", "X", "Y"]:
            qc = add_basis_and_measure(qc0, basis)
            qc.name = f"{dataset_name}_{i:04d}_{basis}"
            circuits.append(qc)
            meta.append({
                "dataset": str(dataset_name),
                "sample_index": int(i),
                "basis": basis,
            })

    return circuits, meta

print("CQRF circuit functions ready. Outputs:", N_OUTPUTS)
print("HUST uses HUST-specific builder matching EXP_CQRF_TinyMLP2_HUST_005.")


CQRF circuit functions ready. Outputs: 57
HUST uses HUST-specific builder matching EXP_CQRF_TinyMLP2_HUST_005.


In [ ]:

# Local statevector dry-run: compare Qiskit CQRF features vs cached simulator R_sim
if RUN_LOCAL_DRY_RUN:
    dry_rows = []

    for ds in ["TJU", "MIT", "HUST"]:
        z = selections[ds]
        n = min(DRY_RUN_SAMPLES_PER_DATASET, len(z["y"]))

        X = z["X_norm"][:n].astype(np.float32)
        R_cached = z["R_sim"][:n].astype(np.float32)

        R_qiskit = local_statevector_cqrf_features(ds, X)

        absdiff = np.abs(R_cached - R_qiskit)

        row = {
            "dataset": ds,
            "n_samples": int(n),
            "mean_abs_feature_diff": float(absdiff.mean()),
            "max_abs_feature_diff": float(absdiff.max()),
            "cached_mean": float(R_cached.mean()),
            "qiskit_mean": float(R_qiskit.mean()),
        }

        dry_rows.append(row)
        print(ds, row)

    dry_df = pd.DataFrame(dry_rows)
    dry_df.to_csv(RESULT_DIR / "local_statevector_dryrun_feature_match.csv", index=False)
    display(dry_df)

    if dry_df["max_abs_feature_diff"].max() > 1e-4:
        print("WARNING: Dry-run feature mismatch is larger than expected. Do NOT run QPU until inspected.")
    else:
        print("Local dry-run passed: Qiskit CQRF features match cached simulator features.")
else:
    print("Local dry run skipped.")


Local dry run skipped.


In [ ]:
# IBM service setup — no token is stored in the notebook
def setup_ibm_service():
    token = os.environ.get('IBM_API_TOKEN')
    if token is None or len(token.strip()) == 0:
        token = getpass.getpass('Paste IBM Quantum API token (hidden, not saved in notebook): ')
    for channel in ['ibm_quantum_platform', 'ibm_quantum']:
        try:
            QiskitRuntimeService.save_account(channel=channel, token=token, overwrite=True)
            service = QiskitRuntimeService(channel=channel)
            print('IBM service initialized with channel:', channel)
            return service
        except Exception as e:
            print('Channel failed:', channel, '|', repr(e)[:200])
    raise RuntimeError('Could not initialize IBM service.')

def choose_backend(service, fixed_backend_name=None):
    if fixed_backend_name:
        backend = service.backend(fixed_backend_name)
    else:
        backend = service.least_busy(min_num_qubits=N_QUBITS, operational=True, simulator=False)
    print('Selected backend:', backend.name)
    try: print('Num qubits:', backend.num_qubits, '| Pending jobs:', backend.status().pending_jobs)
    except Exception: pass
    return backend


In [ ]:
# QPU execution helpers
def get_counts_from_sampler_pub(pub_result):
    data = pub_result.data
    for attr in ['meas','c']:
        if hasattr(data, attr):
            obj = getattr(data, attr)
            if hasattr(obj, 'get_counts'): return obj.get_counts()
    for attr in dir(data):
        if attr.startswith('_'): continue
        try:
            obj = getattr(data, attr)
            if hasattr(obj, 'get_counts'): return obj.get_counts()
        except Exception: pass
    raise RuntimeError(f'Could not extract counts from sampler result: {data}')

def run_qpu_circuits(circuits, meta, backend, shots=SHOTS, batch_size=QPU_BATCH_CIRCUITS):
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    try: sampler = Sampler(mode=backend)
    except TypeError: sampler = Sampler(backend=backend)
    all_counts, job_records, total_wall = [], [], 0.0
    n = len(circuits)
    for start in range(0, n, batch_size):
        end = min(start+batch_size, n)
        print(f'Transpiling/submitting circuits {start}:{end} of {n}...')
        transpiled = pm.run(circuits[start:end])
        t0 = time.time()
        job = sampler.run(transpiled, shots=shots)
        result = job.result()
        wall = time.time() - t0
        total_wall += wall
        job_id = getattr(job, 'job_id', lambda: 'unknown')()
        print(f'  done in {wall:.1f}s | job_id={job_id}')
        job_records.append({'start':start,'end':end,'job_id':str(job_id),'wall_seconds':wall})
        for i in range(len(transpiled)):
            all_counts.append(get_counts_from_sampler_pub(result[i]))
    return all_counts, job_records, total_wall

def counts_to_feature_matrix(counts_list, meta, n_samples):
    grouped = {i:{} for i in range(n_samples)}
    for counts, m in zip(counts_list, meta): grouped[m['sample_index']][m['basis']] = counts
    R = np.zeros((n_samples, N_OUTPUTS), dtype=np.float32)
    for i in range(n_samples):
        if not all(b in grouped[i] for b in ['Z','X','Y']): raise RuntimeError(f'Missing basis counts for sample {i}')
        R[i] = features_from_basis_distributions(grouped[i]['Z'], grouped[i]['X'], grouped[i]['Y'])
    return R

def save_counts_json(path, counts_list, meta):
    serializable = []
    for counts, m in zip(counts_list, meta):
        serializable.append({'meta': m, 'counts': {str(k): int(v) for k,v in counts.items()}})
    with open(path, 'w') as f: json.dump(serializable, f)


In [ ]:

# Pre-QPU safety check: expected circuit count for QPU-150
expected_total_circuits = 0
for ds in ["TJU", "MIT", "HUST"]:
    n_samples = len(selections[ds]["y"])
    n_circuits = n_samples * 3
    expected_total_circuits += n_circuits
    print(f"{ds}: {n_samples} samples -> {n_circuits} circuits")

print("Total circuits:", expected_total_circuits)
assert expected_total_circuits == 1350, f"Expected 1350 circuits, got {expected_total_circuits}"

print("Safety check passed. If RUN_QPU=True, this notebook will submit 1350 circuits.")


TJU: 150 samples -> 450 circuits
MIT: 150 samples -> 450 circuits
HUST: 150 samples -> 450 circuits
Total circuits: 1350
Safety check passed. If RUN_QPU=True, this notebook will submit 1350 circuits.


In [ ]:
# Main QPU block — keep RUN_QPU=False until local dry run passes
if not RUN_QPU:
    print('RUN_QPU is False. No IBM jobs submitted.')
    print('After dry run passes, set RUN_QPU=True and rerun this cell.')
else:
    service = setup_ibm_service()
    backend = choose_backend(service, fixed_backend_name=None)
    qpu_rows, all_job_records = [], []
    for ds in ['TJU','MIT','HUST']:
        print('\n' + '='*100)
        print('QPU dataset:', ds)
        z = selections[ds]
        X, y = z['X_norm'].astype(np.float32), z['y'].astype(np.float32)
        circuits, meta = make_qpu_circuits_for_dataset(ds, X)
        print(f'{ds}: {len(X)} samples -> {len(circuits)} circuits')
        counts_list, job_records, wall = run_qpu_circuits(circuits, meta, backend, shots=SHOTS)
        for jr in job_records:
            jr['dataset'] = ds; all_job_records.append(jr)
        save_counts_json(QPU_FEATURE_DIR / f'{ds}_qpu_counts.json', counts_list, meta)
        R_qpu = counts_to_feature_matrix(counts_list, meta, n_samples=len(X))
        np.savez_compressed(QPU_FEATURE_DIR / f'{ds}_qpu_features.npz', R_qpu=R_qpu, X_norm=X, y=y, shots=np.array(SHOTS), dataset=np.array(ds), backend=np.array(backend.name))
        pred_sim = predict_selection_with_cbr(ds, z['R_sim'].astype(np.float32))
        pred_qpu = predict_selection_with_cbr(ds, R_qpu)
        met_sim, met_qpu = compute_metrics(y, pred_sim), compute_metrics(y, pred_qpu)
        gap = np.abs(pred_qpu - pred_sim)
        feat_gap = np.abs(R_qpu - z['R_sim'].astype(np.float32))
        corr = float(np.corrcoef(R_qpu.reshape(-1), z['R_sim'].astype(np.float32).reshape(-1))[0,1])
        row = {'dataset':ds,'readout':'CBR','samples':int(len(y)),'circuits':int(len(circuits)),'shots':int(SHOTS),'backend':backend.name,
               'sim_MAPE':met_sim['MAPE'],'sim_RMSE':met_sim['RMSE'],'sim_DerivedScore':met_sim['DerivedScore'],
               'qpu_MAPE':met_qpu['MAPE'],'qpu_RMSE':met_qpu['RMSE'],'qpu_DerivedScore':met_qpu['DerivedScore'],
               'mean_abs_prediction_gap':float(gap.mean()),'max_abs_prediction_gap':float(gap.max()),
               'mean_abs_feature_gap':float(feat_gap.mean()),'max_abs_feature_gap':float(feat_gap.max()),
               'feature_correlation':corr,'qpu_wall_seconds':float(wall)}
        qpu_rows.append(row)
        print(pd.DataFrame([row]).to_string(index=False))
        if RUN_RIDGE_CONTROL:
            pred_sim_r = predict_selection_with_ridge(ds, z['R_sim'].astype(np.float32))
            pred_qpu_r = predict_selection_with_ridge(ds, R_qpu)
            met_sim_r, met_qpu_r = compute_metrics(y, pred_sim_r), compute_metrics(y, pred_qpu_r)
            gap_r = np.abs(pred_qpu_r - pred_sim_r)
            row_r = dict(row)
            row_r.update({'readout':'Ridge-control','sim_MAPE':met_sim_r['MAPE'],'sim_RMSE':met_sim_r['RMSE'],'sim_DerivedScore':met_sim_r['DerivedScore'],
                          'qpu_MAPE':met_qpu_r['MAPE'],'qpu_RMSE':met_qpu_r['RMSE'],'qpu_DerivedScore':met_qpu_r['DerivedScore'],
                          'mean_abs_prediction_gap':float(gap_r.mean()),'max_abs_prediction_gap':float(gap_r.max())})
            qpu_rows.append(row_r)
        pred_df = pd.DataFrame({'dataset':ds,'idx':np.arange(len(y)),'y_true':y,'pred_sim_cbr':pred_sim,'pred_qpu_cbr':pred_qpu,'abs_pred_gap':gap})
        if 'cell_ids' in z.keys(): pred_df['cell_id'] = z['cell_ids'].astype(object)
        if 'subset_labels' in z.keys(): pred_df['subset_label'] = z['subset_labels'].astype(object)
        pred_df.to_csv(RESULT_DIR / f'{ds}_qpu_vs_sim_predictions.csv', index=False)
    qpu_df = pd.DataFrame(qpu_rows)
    qpu_df.to_csv(RESULT_DIR / 'qpu_vs_sim_summary.csv', index=False)
    pd.DataFrame(all_job_records).to_csv(RESULT_DIR / 'qpu_job_records.csv', index=False)
    final = {'experiment':EXP_ID,'description':'CQRF + CBR inference-only hardware feasibility spot-check','backend':backend.name,'shots':SHOTS,
             'datasets':['TJU','MIT','HUST'],'samples_per_dataset':150,'readout':'CBR; optional Ridge-control','qpu_training':False,'timestamp':datetime.datetime.now().isoformat()}
    with open(RESULT_DIR / f'{EXP_ID}_final_metadata.json', 'w') as f: json.dump(final, f, indent=2)
    display(qpu_df)
    print('Saved:', RESULT_DIR / 'qpu_vs_sim_summary.csv')


Paste IBM Quantum API token (hidden, not saved in notebook): ··········


qiskit_runtime_service.__init__:WARNING:2026-05-30 18:33:25,624: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


IBM service initialized with channel: ibm_quantum_platform


qiskit_runtime_service.backends:WARNING:2026-05-30 18:33:26,580: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-05-30 18:33:27,748: Using instance: open-instance, plan: open


Selected backend: ibm_marrakesh
Num qubits: 156 | Pending jobs: 0

QPU dataset: TJU
TJU: 150 samples -> 450 circuits
Transpiling/submitting circuits 0:60 of 450...
  done in 215.3s | job_id=d8diqugv14cs73diel3g
Transpiling/submitting circuits 60:120 of 450...
  done in 353.8s | job_id=d8diskq4gq0s73aqbq7g
Transpiling/submitting circuits 120:180 of 450...
  done in 18.9s | job_id=d8divda4gq0s73aqbt70
Transpiling/submitting circuits 180:240 of 450...
  done in 17.8s | job_id=d8dividmdsks73d4056g
Transpiling/submitting circuits 240:300 of 450...
  done in 17.8s | job_id=d8divn0v14cs73dier60
Transpiling/submitting circuits 300:360 of 450...
  done in 19.4s | job_id=d8divrvd0j8c73f4bd0g
Transpiling/submitting circuits 360:420 of 450...
  done in 18.3s | job_id=d8dj00vd0j8c73f4bd70
Transpiling/submitting circuits 420:450 of 450...
  done in 12.1s | job_id=d8dj058v14cs73dierog
dataset readout  samples  circuits  shots       backend  sim_MAPE  sim_RMSE  sim_DerivedScore  qpu_MAPE  qpu_RMSE  qp

,dataset,readout,samples,circuits,shots,backend,sim_MAPE,sim_RMSE,sim_DerivedScore,qpu_MAPE,qpu_RMSE,qpu_DerivedScore,mean_abs_prediction_gap,max_abs_prediction_gap,mean_abs_feature_gap,max_abs_feature_gap,feature_correlation,qpu_wall_seconds
0,TJU,CBR,150,450,512,ibm_marrakesh,0.127309,0.028743,87.269135,1.134011,0.410572,-13.401067,0.340083,1.082472,0.047658,0.431121,0.365209,673.515023
1,TJU,Ridge-control,150,450,512,ibm_marrakesh,0.298036,0.098916,70.196381,1.354178,0.711428,-35.417807,0.571604,1.840140,0.047658,0.431121,0.365209,673.515023
2,MIT,CBR,150,450,512,ibm_marrakesh,0.015101,0.021208,98.489877,0.092272,0.107448,90.772827,0.090395,0.224621,0.045457,0.345790,0.334191,139.183570
3,MIT,Ridge-control,150,450,512,ibm_marrakesh,0.016025,0.021858,98.397464,0.087663,0.105082,91.233707,0.079456,0.286198,0.045457,0.345790,0.334191,139.183570
4,HUST,CBR,150,450,512,ibm_marrakesh,0.021068,0.025324,97.893156,0.198587,0.229147,80.141334,0.180757,0.673581,0.046617,0.273379,0.234936,140.329626
5,HUST,Ridge-control,150,450,512,ibm_marrakesh,0.023838,0.029259,97.616229,0.209623,0.244747,79.037705,0.192889,0.709761,0.046617,0.273379,0.234936,140.329626


Saved: /content/drive/MyDrive/HB/QRC_Battery_SOH/results/EXP-QPU-CQRF-CBR-002-QPU150/qpu_vs_sim_summary.csv


In [ ]:
# Final artifact check
print('Results folder:', RESULT_DIR)
for p in sorted(RESULT_DIR.glob('*')): print(p.name)
print('\nQPU feature folder:', QPU_FEATURE_DIR)
for p in sorted(QPU_FEATURE_DIR.glob('*')): print(p.name)
print('\nCheckpoint folder:', CHECKPOINT_DIR)
for p in sorted(CHECKPOINT_DIR.glob('*')): print(p.name)


Results folder: /content/drive/MyDrive/HB/QRC_Battery_SOH/results/EXP-QPU-CQRF-CBR-002-QPU150
EXP-QPU-CQRF-CBR-002-QPU150_final_metadata.json
HUST_qpu_vs_sim_predictions.csv
MIT_qpu_vs_sim_predictions.csv
TJU_qpu_vs_sim_predictions.csv
cbr_full_test_sanity_metrics.csv
local_statevector_dryrun_feature_match.csv
qpu_job_records.csv
qpu_vs_sim_summary.csv
selected_simulator_metrics.csv

QPU feature folder: /content/drive/MyDrive/HB/QRC_Battery_SOH/features/EXP-QPU-CQRF-CBR-002-QPU150
HUST_qpu_counts.json
HUST_qpu_features.npz
MIT_qpu_counts.json
MIT_qpu_features.npz
TJU_qpu_counts.json
TJU_qpu_features.npz

Checkpoint folder: /content/drive/MyDrive/HB/QRC_Battery_SOH/checkpoints/EXP-QPU-CQRF-CBR-002-QPU150
CQRF_CBR_HUST.pt
CQRF_CBR_MIT.pt
CQRF_CBR_TJU_BLEND.pt
CQRF_CBR_TJU_NCA.pt
CQRF_CBR_TJU_NCM.pt


In [ ]:
from pathlib import Path
import pandas as pd

RESULT_DIR = Path("/content/drive/MyDrive/HB/QRC_Battery_SOH/results/EXP-QPU-CQRF-CBR-002-QPU150")

summary_path = RESULT_DIR / "qpu_vs_sim_summary.csv"
print("Exists:", summary_path.exists())
print("Path:", summary_path)

df = pd.read_csv(summary_path)
cols = [
    "dataset", "readout", "samples", "circuits", "shots", "backend",
    "sim_MAPE", "sim_RMSE", "sim_DerivedScore",
    "qpu_MAPE", "qpu_RMSE", "qpu_DerivedScore",
    "mean_abs_prediction_gap", "mean_abs_feature_gap",
    "feature_correlation", "qpu_wall_seconds"
]
print(df[cols].to_string(index=False))

Exists: True
Path: /content/drive/MyDrive/HB/QRC_Battery_SOH/results/EXP-QPU-CQRF-CBR-002-QPU150/qpu_vs_sim_summary.csv
dataset       readout  samples  circuits  shots       backend  sim_MAPE  sim_RMSE  sim_DerivedScore  qpu_MAPE  qpu_RMSE  qpu_DerivedScore  mean_abs_prediction_gap  mean_abs_feature_gap  feature_correlation  qpu_wall_seconds
    TJU           CBR      150       450    512 ibm_marrakesh  0.127309  0.028743         87.269135  1.134011  0.410572        -13.401067                 0.340083              0.047658             0.365209        673.515023
    TJU Ridge-control      150       450    512 ibm_marrakesh  0.298036  0.098916         70.196381  1.354178  0.711428        -35.417807                 0.571604              0.047658             0.365209        673.515023
    MIT           CBR      150       450    512 ibm_marrakesh  0.015101  0.021208         98.489877  0.092272  0.107448         90.772827                 0.090395              0.045457             0.334191   

In [ ]:
acc = df[[
    "dataset", "readout", "sim_DerivedScore", "qpu_DerivedScore",
    "sim_MAPE", "qpu_MAPE", "feature_correlation"
]].copy()

acc["accuracy_drop_pp"] = acc["qpu_DerivedScore"] - acc["sim_DerivedScore"]
print(acc.to_string(index=False))

dataset       readout  sim_DerivedScore  qpu_DerivedScore  sim_MAPE  qpu_MAPE  feature_correlation  accuracy_drop_pp
    TJU           CBR         87.269135        -13.401067  0.127309  1.134011             0.365209       -100.670202
    TJU Ridge-control         70.196381        -35.417807  0.298036  1.354178             0.365209       -105.614188
    MIT           CBR         98.489877         90.772827  0.015101  0.092272             0.334191         -7.717050
    MIT Ridge-control         98.397464         91.233707  0.016025  0.087663             0.334191         -7.163757
   HUST           CBR         97.893156         80.141334  0.021068  0.198587             0.234936        -17.751822
   HUST Ridge-control         97.616229         79.037705  0.023838  0.209623             0.234936        -18.578524
